## CS-4063: Natural Language Processing — Assignment 2
## Neural NLP Pipeline (BBC Urdu Corpus)

| | |
|---|---|
| **Name** | Munaza Tariq |
| **Roll Number** | i23-2545 |
| **Section** | DS-A |
| **GitHub** | [i23-2545-NLP-Assignment2](https://github.com/i23-2545/i23-2545-NLP-Assignment2) |

---

In [ ]:
import os, re, json, math, random, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# --- Reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Paths -----------------------------------------------------------------
BASE_DIR   = Path('.')
DATA_DIR   = BASE_DIR / 'data'
EMB_DIR    = BASE_DIR / 'embeddings'
MODEL_DIR  = BASE_DIR / 'models'
EMB_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')

## 0. Data Loading Utilities

Load `cleaned.txt`, `raw.txt`, and `Metadata.json` produced by Assignment 1. 
Each article in the `.txt` files is delimited by `[N]` markers.

In [ ]:
# ---------------------------------------------------------------------------
# 0.1  Load text corpus
# ---------------------------------------------------------------------------
def load_corpus(filepath: str) -> list[list[str]]:
    """Load a .txt corpus file into a list of articles.
    
    Each article is a list of sentences (strings).
    Articles are separated by lines matching `[N]`.
    """
    articles: list[list[str]] = []
    current_article: list[str] = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Article delimiter: [1], [2], ...
            if re.match(r'^\[\d+\]$', line):
                if current_article:
                    articles.append(current_article)
                current_article = []
            else:
                current_article.append(line)
    # Don't forget the last article
    if current_article:
        articles.append(current_article)
    
    return articles


def tokenize_article(sentences: list[str]) -> list[str]:
    """Simple whitespace tokenizer for an article's sentences."""
    tokens = []
    for sent in sentences:
        tokens.extend(sent.split())
    return tokens


# Load both corpora
cleaned_articles = load_corpus(DATA_DIR / 'cleaned.txt')
raw_articles     = load_corpus(DATA_DIR / 'raw.txt')

print(f'Cleaned corpus: {len(cleaned_articles)} articles')
print(f'Raw corpus:     {len(raw_articles)} articles')
print(f'Sample cleaned article (first 3 sentences):')
for s in cleaned_articles[0][:3]:
    print(f'  {s}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.2  Load Metadata
# ---------------------------------------------------------------------------
with open(DATA_DIR / 'Metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Metadata entries: {len(metadata)}')
# Show first 5 entries
for k in list(metadata.keys())[:5]:
    print(f'  [{k}] {metadata[k]["title"][:60]}...')

### Vocabulary Builder

Build a vocabulary from the **cleaned** corpus, keeping the **top 10,000 most frequent tokens**. 
All other tokens are mapped to `<UNK>`. We also add a `<PAD>` token for padding sequences.

In [ ]:
# ---------------------------------------------------------------------------
# 0.3  Build Vocabulary
# ---------------------------------------------------------------------------
VOCAB_SIZE = 10000  # top-k tokens
SPECIAL_TOKENS = ['<PAD>', '<UNK>', '<CLS>']

def build_vocabulary(articles: list[list[str]], top_k: int = VOCAB_SIZE):
    """Build word2idx and idx2word mappings from the corpus.
    
    Returns:
        word2idx: dict mapping token -> integer index
        idx2word: list mapping index -> token
        token_counts: Counter of all token frequencies
    """
    # Count all tokens across all articles
    token_counts = Counter()
    for art in articles:
        for sent in art:
            token_counts.update(sent.split())
    
    # Select top-k most frequent tokens
    most_common = token_counts.most_common(top_k)
    
    # Build mappings with special tokens first
    idx2word = list(SPECIAL_TOKENS)
    for token, _ in most_common:
        if token not in SPECIAL_TOKENS:
            idx2word.append(token)
    
    word2idx = {w: i for i, w in enumerate(idx2word)}
    
    return word2idx, idx2word, token_counts


word2idx, idx2word, token_counts = build_vocabulary(cleaned_articles)

PAD_IDX = word2idx['<PAD>']
UNK_IDX = word2idx['<UNK>']
CLS_IDX = word2idx['<CLS>']

print(f'Total unique tokens in corpus: {len(token_counts):,}')
print(f'Vocabulary size (with special): {len(word2idx):,}')
print(f'Coverage: top {VOCAB_SIZE} tokens cover '
      f'{sum(c for _, c in token_counts.most_common(VOCAB_SIZE)):,} / '
      f'{sum(token_counts.values()):,} '
      f'({100*sum(c for _, c in token_counts.most_common(VOCAB_SIZE))/sum(token_counts.values()):.1f}%) occurrences')
print(f'\nTop 20 tokens: {[t for t, _ in token_counts.most_common(20)]}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.4  Helper: encode tokens to indices
# ---------------------------------------------------------------------------
def tokens_to_ids(tokens: list[str], w2i: dict) -> list[int]:
    """Convert a list of tokens to a list of integer IDs."""
    unk = w2i['<UNK>']
    return [w2i.get(t, unk) for t in tokens]


def article_to_ids(article: list[str], w2i: dict) -> list[int]:
    """Tokenize and encode an entire article."""
    tokens = tokenize_article(article)
    return tokens_to_ids(tokens, w2i)


# Quick sanity check
sample_ids = article_to_ids(cleaned_articles[0], word2idx)
print(f'Article 0: {len(sample_ids)} tokens')
print(f'First 20 IDs: {sample_ids[:20]}')
print(f'Decoded back: {[idx2word[i] for i in sample_ids[:20]]}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.5  Topic Category Assignment (keyword-based)
# ---------------------------------------------------------------------------
# We assign each article to one of 5 topic categories using indicative
# keywords from Metadata.json titles (in Urdu).

TOPIC_KEYWORDS = {
    'Politics': [
        'حکومت', 'وزیر', 'اعلیٰ', 'پارلیمنٹ', 'انتخاب', 'سیاسی', 'جمہوریت',
        'قومی', 'اسمبلی', 'صدر', 'وزیراعظم', 'حزب', 'اختلاف', 'عدالت',
        'سپریم', 'کورٹ', 'آئین', 'قانون', 'فیصلہ', 'گرفتار', 'مقدمہ',
        'پولیس', 'دھماکہ', 'حملہ', 'دہشت', 'بم', 'فائرنگ', 'ہلاک',
        'سکیورٹی', 'فوج', 'آپریشن', 'طالبان', 'بلوچستان', 'مسلح',
        'حملے', 'شدت', 'دفاع', 'امن', 'خودکش'
    ],
    'Sports': [
        'کرکٹ', 'میچ', 'ٹیم', 'کھلاڑی', 'ورلڈ', 'کپ', 'فٹبال', 'اسکور',
        'بولنگ', 'بیٹنگ', 'سیریز', 'ٹورنامنٹ', 'فائنل', 'سیمی', 'اوور',
        'رنز', 'وکٹ', 'گول', 'لیگ', 'رونالڈو', 'پاکستان کرکٹ',
        'آئی سی سی', 'کھیل', 'سٹیڈیم', 'اننگز', 'کپتان', 'ایونٹ',
        'سپن', 'بلے', 'باز', 'فاسٹ', 'ریکارڈ'
    ],
    'Economy': [
        'معیشت', 'تجارت', 'بینک', 'بجٹ', 'مہنگائی', 'روپے', 'ڈالر',
        'سرمایہ', 'ٹیکس', 'آمدنی', 'شرح', 'سود', 'قرض', 'پیسے',
        'معاشی', 'اقتصادی', 'مارکیٹ', 'ملازمت', 'تنخواہ', 'مالی',
        'بے روزگاری', 'سرمایہ کاری', 'برآمد', 'درآمد'
    ],
    'International': [
        'امریکہ', 'ٹرمپ', 'چین', 'بھارت', 'انڈیا', 'روس', 'برطانیہ',
        'یورپ', 'اقوام', 'متحدہ', 'سفارتی', 'بین الاقوامی', 'جنگ',
        'غزہ', 'اسرائیل', 'فلسطین', 'یوکرین', 'ناٹو', 'ایران',
        'افغانستان', 'سعودی', 'ترکی', 'عرب', 'مشرق وسطیٰ', 'سفیر',
        'ناروے', 'جرمنی', 'فرانس', 'جاپان', 'کینیڈا', 'آسٹریلیا',
        'دنیا', 'ممالک', 'مغربی', 'سپین'
    ],
    'Health & Society': [
        'ہسپتال', 'بیماری', 'ویکسین', 'صحت', 'تعلیم', 'سکول', 'یونیورسٹی',
        'خواتین', 'بچے', 'آبادی', 'سیلاب', 'زلزلہ', 'موسم', 'ماحول',
        'طوفان', 'آلودگی', 'پانی', 'ادویات', 'ڈاکٹر', 'مریض', 'علاج',
        'سماجی', 'معاشرہ', 'ثقافت', 'تہوار', 'مذہب', 'شادی', 'رسم',
        'پتنگ', 'بسنت', 'عورت', 'لڑکی', 'استحصال', 'جنسی'
    ]
}

def classify_article(article_id: str, meta: dict, articles: list) -> str:
    """Classify an article into one of 5 topic categories.
    
    Uses both the title from metadata AND the article text for matching.
    """
    idx = int(article_id) - 1
    title = meta.get(article_id, {}).get('title', '')
    
    # Combine title + first 5 sentences for richer signal
    text = title
    if 0 <= idx < len(articles):
        text += ' ' + ' '.join(articles[idx][:5])
    
    scores = {}
    for category, keywords in TOPIC_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in text)
        scores[category] = score
    
    best = max(scores, key=scores.get)
    # If no keywords matched, default to 'Health & Society'
    if scores[best] == 0:
        best = 'Health & Society'
    return best


# Classify all articles
article_topics = {}
for art_id in metadata:
    article_topics[art_id] = classify_article(art_id, metadata, cleaned_articles)

# Report distribution
topic_dist = Counter(article_topics.values())
print('\n=== Topic Distribution ===')
for topic, count in topic_dist.most_common():
    print(f'  {topic:25s}: {count:3d} articles ({100*count/len(article_topics):.1f}%)')
print(f'  {"TOTAL":25s}: {len(article_topics):3d}')

In [ ]:
# ---------------------------------------------------------------------------
# 0.6  Save vocabulary (word2idx.json)
# ---------------------------------------------------------------------------
vocab_path = EMB_DIR / 'word2idx.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False, indent=2)

print(f'Saved vocabulary ({len(word2idx):,} entries) to {vocab_path}')

# Also save topic labels for later use
topics_path = DATA_DIR / 'article_topics.json'
with open(topics_path, 'w', encoding='utf-8') as f:
    json.dump(article_topics, f, ensure_ascii=False, indent=2)

print(f'Saved article topics to {topics_path}')

---
## Part 1 — Word Embeddings (25 Marks)

### 1.1 TF-IDF Weighting (4 Marks)

- Build a **term-document matrix** from `cleaned.txt`
- Vocabulary: top 10,000 tokens; others → `<UNK>`
- Formula: `TF-IDF(w, d) = TF(w, d) × log(N / (1 + df(w)))`
- Save as `tfidf_matrix.npy`
- Report **top-10 discriminative words** per topic

In [ ]:
# ===========================================================================
# 1.1  TF-IDF Weighting
# ===========================================================================

N = len(cleaned_articles)   # total number of documents (239)
V = len(word2idx)           # vocabulary size (10003 with special tokens)
print(f'Documents (N): {N}')
print(f'Vocabulary (V): {V}')

# -- STEP 1: Build Term-Frequency Matrix ----------------------------------
# tf_matrix[doc][word] = how many times 'word' appears in 'doc'
# Shape: (239, 10003)

print('\nStep 1: Building term-frequency matrix...')
tf_matrix = np.zeros((N, V), dtype=np.float64)

for doc_idx, article in enumerate(cleaned_articles):
    for sentence in article:
        for token in sentence.split():
            # If token is in vocabulary, use its index; otherwise use <UNK>
            if token in word2idx:
                word_idx = word2idx[token]
            else:
                word_idx = word2idx['<UNK>']
            tf_matrix[doc_idx][word_idx] += 1

print(f'  TF matrix shape: {tf_matrix.shape}')
print(f'  Non-zero entries: {np.count_nonzero(tf_matrix):,}')

# -- STEP 2: Compute Document Frequency -----------------------------------
# df[word] = in how many documents does this word appear?

print('\nStep 2: Computing document frequencies...')
df = np.zeros(V, dtype=np.float64)
for word_idx in range(V):
    df[word_idx] = np.sum(tf_matrix[:, word_idx] > 0)

print(f'  Words in all {N} docs: {int(np.sum(df == N))}')
print(f'  Words in only 1 doc:  {int(np.sum(df == 1))}')

# -- STEP 3: Compute TF-IDF ------------------------------------------------
# Formula: TF-IDF(w, d) = TF(w, d) * log( N / (1 + df(w)) )

print('\nStep 3: Computing TF-IDF weights...')
idf = np.log(N / (1 + df))                    # shape: (V,)
tfidf_matrix = tf_matrix * idf[np.newaxis, :]  # shape: (N, V)

print(f'  TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  Value range: [{tfidf_matrix.min():.4f}, {tfidf_matrix.max():.4f}]')
print(f'  Mean: {tfidf_matrix.mean():.6f}')

# -- STEP 4: Save to file --------------------------------------------------
tfidf_path = EMB_DIR / 'tfidf_matrix.npy'
np.save(tfidf_path, tfidf_matrix)
print(f'\nSaved TF-IDF matrix to {tfidf_path}')

# Quick verification
loaded = np.load(tfidf_path)
print(f'  Verified: loaded shape = {loaded.shape}')

In [ ]:
# ===========================================================================
# 1.1 (cont.)  Top-10 Most Discriminative Words Per Topic Category
# ===========================================================================
# For each topic, find words with the highest AVERAGE TF-IDF score
# across all documents belonging to that topic.

# Group document indices by their topic
topic_to_docs = {}
for art_id, topic in article_topics.items():
    doc_idx = int(art_id) - 1
    if doc_idx < N:
        if topic not in topic_to_docs:
            topic_to_docs[topic] = []
        topic_to_docs[topic].append(doc_idx)

# For each topic, compute average TF-IDF and find top-10 words
for topic in sorted(topic_to_docs.keys()):
    doc_indices = topic_to_docs[topic]
    
    # Get the TF-IDF rows for this topic's documents
    topic_tfidf = tfidf_matrix[doc_indices, :]   # shape: (num_docs, V)
    
    # Average TF-IDF for each word across this topic's docs
    avg_tfidf = topic_tfidf.mean(axis=0)         # shape: (V,)
    
    # Sort words by average TF-IDF (highest first)
    sorted_indices = np.argsort(avg_tfidf)[::-1]
    
    print(f'\n{"="*50}')
    print(f'Topic: {topic} ({len(doc_indices)} articles)')
    print(f'{"="*50}')
    print(f'{"Rank":<6} {"Word":<25} {"Avg TF-IDF":<12}')
    print(f'{"-"*45}')
    
    count = 0
    for idx in sorted_indices:
        # Skip special tokens (<PAD>=0, <UNK>=1, <CLS>=2)
        if idx < len(SPECIAL_TOKENS):
            continue
        word = idx2word[idx]
        score = avg_tfidf[idx]
        count += 1
        print(f'{count:<6} {word:<25} {score:.4f}')
        if count >= 10:
            break

---
### 1.2 Pointwise Mutual Information — PMI (5 Marks)

- Build a **word-word co-occurrence matrix** (window k=5)
- Compute **PPMI**: `max(0, log2( P(w1,w2) / (P(w1) * P(w2)) ))`
- Save as `ppmi_matrix.npy`
- 2D **t-SNE visualization** of the 200 most frequent tokens
- Report **top-5 nearest neighbours** by cosine similarity for 12 query words

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.2  PPMI Co-occurrence Matrix
# ═══════════════════════════════════════════════════════════════════════════
import time
from sklearn.manifold import TSNE

K_WINDOW = 5

# 1. Build a flat list of all token IDs in the corpus for easier context windowing
all_token_ids = []
for article in cleaned_articles:
    for sentence in article:
        for token in sentence.split():
            all_token_ids.append(word2idx.get(token, UNK_IDX))

print(f'Total tokens in corpus: {len(all_token_ids):,}')

# Count global frequencies (to calculate probabilities later)
token_freq = np.zeros(V, dtype=np.float64)
for tid in all_token_ids:
    token_freq[tid] += 1

# 2. Build co-occurrence matrix
print(f'\nBuilding co-occurrence matrix (window k={K_WINDOW})...')
cooccur = np.zeros((V, V), dtype=np.float64)
total_pairs = 0

for i in range(len(all_token_ids)):
    center = all_token_ids[i]
    # Look k tokens to the left and right
    start = max(0, i - K_WINDOW)
    end = min(len(all_token_ids), i + K_WINDOW + 1)
    
    for j in range(start, end):
        if i == j:
            continue
        context = all_token_ids[j]
        cooccur[center][context] += 1
        total_pairs += 1

print(f'  Total co-occurrence pairs: {total_pairs:,}')

# 3. Compute PPMI
print('\nComputing PPMI matrix...')
total_cooccur = cooccur.sum()
row_sums = cooccur.sum(axis=1)
col_sums = cooccur.sum(axis=0)

ppmi_matrix = np.zeros((V, V), dtype=np.float64)
nonzero_rows, nonzero_cols = np.nonzero(cooccur)

for w1, w2 in zip(nonzero_rows, nonzero_cols):
    if row_sums[w1] > 0 and col_sums[w2] > 0:
        # P(w1, w2) = cooccur[w1, w2] / total_cooccur
        # P(w1) = row_sums[w1] / total_cooccur
        # P(w2) = col_sums[w2] / total_cooccur
        pmi = np.log2((cooccur[w1, w2] * total_cooccur) / (row_sums[w1] * col_sums[w2]))
        ppmi_matrix[w1, w2] = max(0, pmi)

print(f'  PPMI matrix shape: {ppmi_matrix.shape}')
print(f'  Non-zero PPMI entries: {np.count_nonzero(ppmi_matrix):,}')

# 4. Save to file
ppmi_path = EMB_DIR / 'ppmi_matrix.npy'
np.save(ppmi_path, ppmi_matrix)
print(f'\nSaved PPMI matrix to {ppmi_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.2 (cont.)  t-SNE Visualization & Nearest Neighbours
# ═══════════════════════════════════════════════════════════════════════════
print('Generating t-SNE visualization for top 200 tokens...')

# Get 200 most frequent tokens (excluding special tokens)
freq_order = np.argsort(token_freq)[::-1]
top200 = []
for idx in freq_order:
    if idx >= len(SPECIAL_TOKENS):
        top200.append(idx)
    if len(top200) >= 200:
        break

# Extract vectors
vectors_200 = ppmi_matrix[top200, :]

# Assign semantic categories for coloring
SEMANTIC_CATS = {
    'Politics': ['حکومت','وزیر','پارلیمنٹ','سیاسی','عدالت','قانون','پولیس','فوج',
                 'حملہ','فائرنگ','سکیورٹی','مسلح','دہشت','انتخاب','صدر','آئین'],
    'Sports': ['کرکٹ','میچ','ٹیم','کھلاڑی','ورلڈ','کپ','بولنگ','بیٹنگ','سیریز',
               'ٹورنامنٹ','فائنل','رنز','وکٹ','کھیل','سٹیڈیم','کپتان'],
    'Geography': ['پاکستان','انڈیا','امریکہ','چین','برطانیہ','ایران','افغانستان',
                  'سعودی','کراچی','لاہور','اسلام','آباد','بلوچستان','کوئٹہ','لندن']
}

colors_200 = []
for idx in top200:
    word = idx2word[idx]
    assigned = 'Other'
    for cat, keywords in SEMANTIC_CATS.items():
        if word in keywords:
            assigned = cat
            break
    colors_200.append(assigned)

color_map = {'Politics': '#e74c3c', 'Sports': '#2ecc71', 'Geography': '#3498db', 'Other': '#95a5a6'}

# Run t-SNE
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000)
coords = tsne.fit_transform(vectors_200)

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
for cat, color in color_map.items():
    mask = [c == cat for c in colors_200]
    ax.scatter(coords[mask, 0], coords[mask, 1], c=color, label=cat, alpha=0.7, s=50)

# Add labels to a few points to avoid clutter
import matplotlib.font_manager as fm
try:
    prop = fm.FontProperties(family='Arial', size=9)
    for i, idx in enumerate(top200):
        if colors_200[i] != 'Other':
            ax.annotate(idx2word[idx], (coords[i, 0], coords[i, 1]), fontproperties=prop, alpha=0.8)
except:
    pass # Ignore missing font for Urdu display

ax.set_title('t-SNE Visualization of Top 200 Tokens (PPMI Vectors)', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(EMB_DIR / 'tsne_ppmi.png', dpi=150)
plt.show()
print('Saved t-SNE plot to embeddings/tsne_ppmi.png')

# ── Nearest Neighbours ────────────────────────────────────────────────
print('\nTop-5 Nearest Neighbours by Cosine Similarity:')

def cosine_sim(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return np.dot(v1, v2) / (n1 * n2) if n1 > 0 and n2 > 0 else 0.0

query_words = ['پاکستان', 'حکومت', 'کرکٹ', 'پولیس', 'امریکہ', 'عدالت',
               'فوج', 'تعلیم', 'صحت', 'معیشت', 'سعودی', 'خواتین']

for qw in query_words:
    if qw not in word2idx:
        continue
    q_vec = ppmi_matrix[word2idx[qw]]
    sims = []
    for i in range(len(SPECIAL_TOKENS), V):
        if i == word2idx[qw]:
            continue
        sims.append((idx2word[i], cosine_sim(q_vec, ppmi_matrix[i])))
    
    sims.sort(key=lambda x: -x[1])
    top5 = [f"{w} ({s:.2f})" for w, s in sims[:5]]
    print(f"  {qw:<10}: {', '.join(top5)}")

---
### 2.1 Skip-gram Word2Vec Implementation (9 Marks)

- Train Skip-gram model from scratch
- Center (V) and Context (U) embedding matrices
- Noise distribution: `Pn(w) ∝ f(w)^(3/4)`, K=10 noise samples
- Binary cross-entropy loss over context window k=5
- Train for 5 epochs, batch size >= 512, lr=0.001
- Save final embeddings `½(V + U)` as `embeddings_w2v.npy`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.1  Skip-gram Word2Vec Training
# ═══════════════════════════════════════════════════════════════════════════

EMB_DIM = 100
WINDOW  = 5
NEG_K   = 10
LR      = 0.001
EPOCHS  = 5
BATCH   = 512

print('Building training pairs (center, context)...')
pairs = []
for i in range(len(all_token_ids)):
    center = all_token_ids[i]
    start = max(0, i - WINDOW)
    end = min(len(all_token_ids), i + WINDOW + 1)
    for j in range(start, end):
        if i != j:
            pairs.append((center, all_token_ids[j]))

pairs = np.array(pairs, dtype=np.int64)
print(f'  Total training pairs: {len(pairs):,}')

# ── 1. Create Negative Sampling Table ──────────────────────────────────────
# Pre-compute a large flat array containing word indices with probability proportional
# to f(w)^(3/4). This allows extremely fast negative sampling during training.

print('\nBuilding unigram noise table...')
freq_34 = np.power(token_freq, 0.75)
noise_dist = freq_34 / freq_34.sum()

TABLE_SIZE = 100_000_000
unigram_table = np.zeros(TABLE_SIZE, dtype=np.int64)

offset = 0
for i in range(V):
    count = int(noise_dist[i] * TABLE_SIZE)
    if count > 0:
        unigram_table[offset : offset+count] = i
        offset += count

# Fill any remainder with UNK
unigram_table[offset:] = UNK_IDX
unigram_table = torch.tensor(unigram_table, dtype=torch.long, device=DEVICE)
print('  Noise table ready.')

# ── 2. Dataset & DataLoader ────────────────────────────────────────────────
class SkipGramDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        return self.pairs[idx][0], self.pairs[idx][1]

loader = DataLoader(SkipGramDataset(pairs), batch_size=BATCH, shuffle=True)
print(f'  Batches per epoch: {len(loader)}')

# ── 3. Skip-gram Model ─────────────────────────────────────────────────────
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.center_emb  = nn.Embedding(vocab_size, emb_dim)
        self.context_emb = nn.Embedding(vocab_size, emb_dim)
        
        # Initialize with small random values
        initrange = 0.5 / emb_dim
        self.center_emb.weight.data.uniform_(-initrange, initrange)
        self.context_emb.weight.data.uniform_(-initrange, initrange)

    def forward(self, center_ids, context_ids, neg_ids):
        # Embeddings: (batch, dim)
        v_c = self.center_emb(center_ids)
        u_o = self.context_emb(context_ids)
        u_neg = self.context_emb(neg_ids) # (batch, K, dim)
        
        # Positive score: log sigmoid(u_o . v_c)
        pos_score = torch.sum(v_c * u_o, dim=1)
        pos_loss = -F.logsigmoid(pos_score)
        
        # Negative score: sum of log sigmoid(-u_neg . v_c)
        neg_score = torch.bmm(u_neg, v_c.unsqueeze(2)).squeeze(2)
        neg_loss = -F.logsigmoid(-neg_score).sum(dim=1)
        
        return (pos_loss + neg_loss).mean()

model = SkipGramModel(V, EMB_DIM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

# ── 4. Training Loop ───────────────────────────────────────────────────────
print(f'\nTraining Skip-gram (d={EMB_DIM}, k={WINDOW}, K={NEG_K}, lr={LR})...')

loss_history = []

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    t0 = time.time()
    
    for i, (centers, contexts) in enumerate(loader):
        centers = centers.to(DEVICE)
        contexts = contexts.to(DEVICE)
        
        # Extremely fast negative sampling from the unigram table
        rand_indices = torch.randint(0, TABLE_SIZE, (centers.size(0), NEG_K), device=DEVICE)
        neg_samples = unigram_table[rand_indices]
        
        loss = model(centers, contexts, neg_samples)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(loader)
    loss_history.append(avg_loss)
    elapsed = time.time() - t0
    print(f'  Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s')

# ── 5. Plot Loss & Save Embeddings ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, EPOCHS + 1), loss_history, 'o-', color='#e74c3c', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Skip-gram Word2Vec Training Loss', fontsize=14)
ax.set_xticks(range(1, EPOCHS + 1))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(EMB_DIR / 'w2v_loss.png', dpi=150)
plt.show()

# Final embeddings = average of center and context matrices: 0.5 * (V + U)
with torch.no_grad():
    V_mat = model.center_emb.weight.cpu().numpy()
    U_mat = model.context_emb.weight.cpu().numpy()
    embeddings = 0.5 * (V_mat + U_mat)

emb_path = EMB_DIR / 'embeddings_w2v.npy'
np.save(emb_path, embeddings)
print(f'\nSaved embeddings to {emb_path}')

---
### 2.2 Evaluation (7 Marks)

In [ ]:
# PLACEHOLDER - Chunk 4: Embedding Evaluation
# Will be implemented in the next commit.